# Persist and retrieve case memory

This notebook saves a few verified practice records to a local file, reloads them, and retrieves only the records relevant to a new question.

## Step 1: Import the persistence and message helpers

The next cell imports standard-Python file and JSON tools, plus AgentScope message classes used to turn retrieved records into agent-ready context.

In [ ]:
import json
from pathlib import Path

from agentscope.message import Msg, TextBlock

## Step 2: Choose a durable memory location

The next cell creates a local `data` folder and names the JSON file that will outlive this notebook session.

In [ ]:
data_directory = Path("data")
data_directory.mkdir(exist_ok=True)
memory_path = data_directory / "practice_case_memory.json"
print(f"Memory file: {memory_path.resolve()}")

## Step 3: Create verified practice records

The next cell defines the small records that are safe to retain: each includes a case identifier, a fact, and its source. These records are fictional examples, not real evidence.

In [ ]:
practice_records = [
    {
        "case_id": "INC-204",
        "fact": "The workstation contacted 192.0.2.44 forty-three times at 09:14 UTC.",
        "source": "practice network alert",
    },
    {
        "case_id": "INC-204",
        "fact": "The local practice list marks 192.0.2.44 as suspicious; this is not proof of malicious activity.",
        "source": "practice IP record",
    },
    {
        "case_id": "INC-204",
        "fact": "The available alert does not identify the process that made the contacts.",
        "source": "practice network alert",
    },
]
print(f"Prepared {len(practice_records)} verified practice records.")

## Step 4: Save the memory

The next cell writes the structured records to JSON. Saving selected records is different from saving every conversation message: it is a deliberate retention decision.

In [ ]:
with memory_path.open("w", encoding="utf-8") as memory_file:
    json.dump(practice_records, memory_file, indent=2)

print(f"Saved {len(practice_records)} records to disk.")

## Step 5: Load the memory in a new session

The next cell reloads the JSON file into a new variable. This simulates returning to the case after the original notebook variables are gone.

In [ ]:
with memory_path.open(encoding="utf-8") as memory_file:
    stored_records = json.load(memory_file)

print(f"Loaded {len(stored_records)} durable records.")

## Step 6: Retrieve records relevant to one question

The next cell uses a simple keyword-overlap rule. Its limits are intentional: students can inspect exactly why records are returned before studying semantic RAG systems.

In [ ]:
def retrieve_records(question: str, records: list[dict]) -> list[dict]:
    """Return records whose case ID or fact shares a keyword with the question."""
    keywords = set(question.lower().replace("?", "").split())
    matches = []
    for record in records:
        searchable_text = f"{record['case_id']} {record['fact']}".lower()
        if any(keyword in searchable_text for keyword in keywords):
            matches.append(record)
    return matches

question = "What does INC-204 show about 192.0.2.44?"
retrieved_records = retrieve_records(question, stored_records)

print(f"Retrieved {len(retrieved_records)} record(s):")
for record in retrieved_records:
    print(f"- {record['fact']} (source: {record['source']})")

## Step 7: Build bounded context for an agent

The next cell formats only the retrieved records as an AgentScope message. An agent receiving this message should use these facts, cite their limits, and avoid inventing details not present in the records.

In [ ]:
retrieved_text = "\n".join(
    f"- {record['fact']} Source: {record['source']}."
    for record in retrieved_records
)

retrieved_context = Msg(
    name="memory",
    role="user",
    content=[TextBlock(text=(
        "Use only the following retrieved practice records when answering. "
        "State when the records are insufficient.\n"
        f"{retrieved_text}"
    ))],
)

print(retrieved_context.content[0].text)

## Step 8: Checkpoint

Add an `INC-319` record to `practice_records`, save it, and run the retrieval again for `INC-204`. Confirm that the retrieved context remains focused on the requested case.